In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-obp qiskit-addon-utils rustworkx
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Wsteczna propagacja operatora (OBP) do estymacji wartości oczekiwanych

import TutorialFeedback from '@site/src/components/TutorialFeedback';



*Szacowane zużycie zasobów: 4 minuty na procesorze Heron r3 (UWAGA: To tylko szacunek. Rzeczywisty czas wykonania może się różnić.)*
## Cele kształcenia
Po ukończeniu tego samouczka użytkownicy powinni rozumieć:
- Jak używać [`qiskit-addon-obp`](https://github.com/Qiskit/qiskit-addon-obp) do redukcji głębokości obwodu kwantowego kosztem zwiększonej liczby wykonań obwodu
- Jak używać [`qiskit-addon-utils`](https://github.com/Qiskit/qiskit-addon-utils) do konstruowania hamiltonianów XYZ i odpowiadających im obwodów ewolucji czasowej

## Wymagania wstępne
Zalecamy, aby użytkownicy zapoznali się z następującymi tematami przed przystąpieniem do tego samouczka:
- Używanie prymitywu [Estimator](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) do obliczania wartości oczekiwanych obserwowalnej

## Wprowadzenie
Wsteczna propagacja operatora to technika polegająca na wchłanianiu operacji z końca obwodu kwantowego do mierzonej obserwowalnej, co ogólnie zmniejsza głębokość obwodu kosztem dodatkowych składników w obserwowalnej. Celem jest wsteczne propagowanie jak największej części obwodu bez dopuszczenia do nadmiernego wzrostu obserwowalnej. Implementacja oparta na Qiskit jest dostępna w rozszerzeniu OBP Qiskit. Więcej informacji można znaleźć w odpowiedniej [dokumentacji](https://qiskit.github.io/qiskit-addon-obp/).

Rozważmy przykładowy obwód, dla którego mierzona jest obserwowalna $O = \sum_P c_P P$, gdzie $P$ są Paulimi, a $c_P$ są współczynnikami. Oznaczmy obwód jako pojedynczą unitarną $U$, którą można logicznie podzielić na $U = U_C U_Q$, jak pokazano na rysunku poniżej.

![Diagram obwodu przedstawiający Uq a następnie Uc](../docs/images/tutorials/improving-estimation-of-expectation-values-with-operator-backpropagation/logical-partitioning.avif)

Wsteczna propagacja operatora wchłania unitarną $U_C$ do obserwowalnej poprzez jej ewolucję jako $O' = U_C^{\dagger}OU_C = \sum_P c_P U_C^{\dagger}PU_C$. Innymi słowy, część obliczenia jest wykonywana klasycznie poprzez ewolucję obserwowalnej z $O$ do $O'$. Pierwotny problem można teraz sformułować na nowo jako pomiar obserwowalnej $O'$ dla nowego obwodu o mniejszej głębokości, którego unitarna to $U_Q$.

Unitarna $U_C$ jest reprezentowana jako liczba przedziałów $U_C = U_S U_{S-1}...U_2U_1$. Istnieje wiele sposobów definiowania przedziału. Na przykład, w powyższym przykładowym obwodzie każda warstwa bramek $R_{zz}$ i każda warstwa bramek $R_x$ może być traktowana jako osobny przedział. Wsteczna propagacja polega na klasycznym obliczaniu $O' = \Pi_{s=1}^S \sum_P c_P U_s^{\dagger} P U_s$. Każdy przedział $U_s$ można przedstawić jako $U_s = exp(\frac{-i\theta_s P_s}{2})$, gdzie $P_s$ jest $n$-qubitowym Paulim, a $\theta_s$ jest skalarem. Łatwo można sprawdzić, że

$$
U_s^{\dagger} P U_s = P \qquad \text{if} ~[P,P_s] = 0,
$$

$$
U_s^{\dagger} P U_s = \qquad cos(\theta_s)P + i sin(\theta_s)P_sP \qquad \text{if} ~{P,P_s} = 0
$$

W powyższym przykładzie, jeśli ${P,P_s} = 0$, musimy wykonać dwa obwody kwantowe zamiast jednego, aby obliczyć wartość oczekiwaną. Dlatego wsteczna propagacja może zwiększyć liczbę składników w obserwowalnej, co prowadzi do większej liczby wykonań obwodu. Jednym ze sposobów umożliwienia głębszej wstecznej propagacji do obwodu przy jednoczesnym zapobieganiu nadmiernemu wzrostowi operatora jest obcinanie składników o małych współczynnikach zamiast dodawania ich do operatora. Na przykład w powyższym przykładzie można zdecydować się na obcięcie składnika zawierającego $P_sP$, pod warunkiem że $\theta_s$ jest wystarczająco mały. Obcinanie składników może skutkować mniejszą liczbą obwodów kwantowych do wykonania, jednak wiąże się z pewnym błędem w końcowym obliczeniu wartości oczekiwanej proporcjonalnym do wielkości współczynników obciętych składników.
## Wymagania
Przed rozpoczęciem tego samouczka upewnij się, że masz zainstalowane następujące pakiety:

- Qiskit SDK v2.0 lub nowszy, z obsługą [wizualizacji](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 lub nowszy (`pip install qiskit-ibm-runtime`)
- Rozszerzenie OBP Qiskit 0.3 lub nowsze (`pip install qiskit-addon-obp`)
- Qiskit addon utils 0.3 lub nowsze (`pip install qiskit-addon-utils`)
## Konfiguracja

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.primitives import StatevectorEstimator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import CouplingMap
from qiskit.synthesis import LieTrotter

from qiskit_addon_utils.problem_generators import generate_xyz_hamiltonian
from qiskit_addon_utils.problem_generators import (
    generate_time_evolution_circuit,
)
from qiskit_addon_utils.slicing import slice_by_depth, combine_slices
from qiskit_addon_obp.utils.simplify import OperatorBudget
from qiskit_addon_obp import backpropagate
from qiskit_addon_obp.utils.truncating import setup_budget

from rustworkx.visualization import graphviz_draw

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2, EstimatorOptions

## Przykład na małą skalę z symulatorem
Ten samouczek implementuje [wzorzec Qiskit](/guides/intro-to-patterns) do symulacji dynamiki kwantowej łańcucha spinów Heisenberga przy użyciu [rozszerzenia OBP Qiskit](https://github.com/Qiskit/qiskit-addon-obp). Zauważ, że w symulatorze bez szumów wartość oczekiwana uzyskana z wsteczną propagacją i bez niej będzie taka sama.
### Krok 1: Odwzorowanie klasycznych danych wejściowych na problem kwantowy
#### Odwzorowanie ewolucji czasowej kwantowego modelu Heisenberga na eksperyment kwantowy
Najpierw użyjemy funkcji [`generate_xyz_hamiltonian`](https://docs.quantum.ibm.com/api/qiskit-addon-utils/problem-generators#generate_xyz_hamiltonian) z `qiskit-addon-utils` do wygenerowania hamiltonianu podobnego do Heisenberga na danym grafie połączeń. Tym grafem może być [rustworkx.PyGraph](https://www.rustworkx.org/apiref/rustworkx.PyGraph.html) lub [CouplingMap](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.CouplingMap). Poniżej użyjemy liniowej mapy sprzężeń `CouplingMap` złożonej z 10 qubitów.

In [2]:
num_qubits = 10
layout = [(i - 1, i) for i in range(1, num_qubits)]

# Instantiate a CouplingMap object
coupling_map = CouplingMap(layout)
graphviz_draw(coupling_map.graph, method="circo")

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif" alt="Output of the previous code cell" />

![Wyjście poprzedniej komórki kodu](../docs/images/tutorials/operator-back-propagation/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif)

Następnie generujemy operator Pauliego modelujący hamiltoniana Heisenberga XYZ:

$$
{\hat{\mathcal{H}}_{XYZ} = \sum_{(j,k)\in E} (J_{x} \sigma_j^{x} \sigma_{k}^{x} + J_{y} \sigma_j^{y} \sigma_{k}^{y} + J_{z} \sigma_j^{z} \sigma_{k}^{z}) + \sum_{j\in V} (h_{x} \sigma_j^{x} + h_{y} \sigma_j^{y} + h_{z} \sigma_j^{z}),}
$$

gdzie $G(V,E)$ jest grafem mapy sprzężeń. W tym samouczku użyliśmy $J_x, J_y, J_z$ równych odpowiednio $\frac{\pi}{8}, \frac{\pi}{4}, \frac{\pi}{2}$, oraz $h_x, h_y, h_z$ równych odpowiednio $\frac{\pi}{3}, \frac{\pi}{6}, \frac{\pi}{9}$.

In [3]:
# Get a qubit operator describing the Heisenberg XYZ model
hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(np.pi / 8, np.pi / 4, np.pi / 2),
    ext_magnetic_field=(np.pi / 3, np.pi / 6, np.pi / 9),
)
print(hamiltonian)

SparsePauliOp(['IIIIIIIXXI', 'IIIIIIIYYI', 'IIIIIIIZZI', 'IIIIIXXIII', 'IIIIIYYIII', 'IIIIIZZIII', 'IIIXXIIIII', 'IIIYYIIIII', 'IIIZZIIIII', 'IXXIIIIIII', 'IYYIIIIIII', 'IZZIIIIIII', 'IIIIIIIIXX', 'IIIIIIIIYY', 'IIIIIIIIZZ', 'IIIIIIXXII', 'IIIIIIYYII', 'IIIIIIZZII', 'IIIIXXIIII', 'IIIIYYIIII', 'IIIIZZIIII', 'IIXXIIIIII', 'IIYYIIIIII', 'IIZZIIIIII', 'XXIIIIIIII', 'YYIIIIIIII', 'ZZIIIIIIII', 'IIIIIIIIIX', 'IIIIIIIIIY', 'IIIIIIIIIZ', 'IIIIIIIIXI', 'IIIIIIIIYI', 'IIIIIIIIZI', 'IIIIIIIXII', 'IIIIIIIYII', 'IIIIIIIZII', 'IIIIIIXIII', 'IIIIIIYIII', 'IIIIIIZIII', 'IIIIIXIIII', 'IIIIIYIIII', 'IIIIIZIIII', 'IIIIXIIIII', 'IIIIYIIIII', 'IIIIZIIIII', 'IIIXIIIIII', 'IIIYIIIIII', 'IIIZIIIIII', 'IIXIIIIIII', 'IIYIIIIIII', 'IIZIIIIIII', 'IXIIIIIIII', 'IYIIIIIIII', 'IZIIIIIIII', 'XIIIIIIIII', 'YIIIIIIIII', 'ZIIIIIIIII'],
              coeffs=[0.39269908+0.j, 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j,
 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j, 0.78539816+0.j,
 1.57079633+0.j, 0.39269908+0.j, 0.

From the qubit operator, we can generate a quantum circuit which models its time evolution. We have used [`generate_time_evolution_circuit`](/docs/api/qiskit-addon-utils/problem-generators#generate_time_evolution_circuit) with Lie Trotter decomposition to construct the time evolution circuit.

In [4]:
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=0.2,
    synthesis=LieTrotter(reps=2),
)
circuit.draw("mpl", style="iqp", fold=-1)

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/5208e0a8-0.avif" alt="Output of the previous code cell" />

Na podstawie operatora qubitowego możemy wygenerować obwód kwantowy modelujący jego ewolucję czasową. Użyliśmy [`generate_time_evolution_circuit`](https://docs.quantum.ibm.com/api/qiskit-addon-utils/problem-generators#generate_time_evolution_circuit) z dekompozycją Lie Trottera do skonstruowania obwodu ewolucji czasowej.

In [5]:
slices = slice_by_depth(circuit, max_slice_depth=1)
print(f"Separated the circuit into {len(slices)} slices.")

Separated the circuit into 18 slices.


![Wyjście poprzedniej komórki kodu](../docs/images/tutorials/operator-back-propagation/extracted-outputs/5208e0a8-0.avif)

### Krok 2: Optymalizacja problemu pod kątem wykonania na sprzęcie kwantowym
#### Tworzenie przedziałów obwodu do wstecznej propagacji
Funkcja `backpropagate` propaguje wstecz całe przedziały obwodu naraz. Dlatego wybór sposobu podziału na przedziały może mieć wpływ na efektywność wstecznej propagacji dla danego problemu. Tutaj pogrupujemy bramki tego samego typu w przedziały przy użyciu funkcji [`slice_by_depth`](https://docs.quantum.ibm.com/api/qiskit-addon-utils/slicing#slice_by_depth).

Bardziej szczegółowe omówienie podziału obwodu na przedziały można znaleźć w tym [przewodniku](https://qiskit.github.io/qiskit-addon-utils/how_tos/create_circuit_slices.html) pakietu [`qiskit-addon-utils`](https://github.com/Qiskit/qiskit-addon-utils).

In [6]:
op_budget = OperatorBudget(max_qwc_groups=8)

#### Backpropagate slices from the circuit

First we specify the observable to be $M_Z = \frac{1}{N} \sum_{i=1}^N \langle Z_i \rangle$, $N$ being the number of qubits. We will backpropagate slices from the time-evolution circuit until the terms in the observable can no longer be combined into eight or fewer qubit-wise commuting Pauli groups.

In [7]:
observable = SparsePauliOp.from_sparse_list(
    [("Z", [i], 1 / num_qubits) for i in range(num_qubits)],
    num_qubits=num_qubits,
)
observable

SparsePauliOp(['IIIIIIIIIZ', 'IIIIIIIIZI', 'IIIIIIIZII', 'IIIIIIZIII', 'IIIIIZIIII', 'IIIIZIIIII', 'IIIZIIIIII', 'IIZIIIIIII', 'IZIIIIIIII', 'ZIIIIIIIII'],
              coeffs=[0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j,
 0.1+0.j, 0.1+0.j])

#### Ograniczanie wzrostu operatora podczas wstecznej propagacji
Podczas wstecznej propagacji liczba składników w operatorze będzie generalnie szybko zbliżać się do $2^L$, gdzie $L$ jest liczbą przedziałów. Gdy dwa składniki w operatorze nie komutują qubitowo, potrzebujemy osobnych obwodów do uzyskania odpowiadających im wartości oczekiwanych. Na przykład, jeśli mamy dwuqubitową obserwowalną $O = 0.1 XX + 0.3 IZ - 0.5 IX$, to ponieważ $[XX,IX] = 0$, pomiar w jednej bazie wystarcza do obliczenia wartości oczekiwanych dla tych dwóch składników. Jednak $IZ$ antykomutuje z pozostałymi dwoma składnikami, więc potrzebujemy osobnego pomiaru bazowego do obliczenia wartości oczekiwanej $IZ$. Innymi słowy, potrzebujemy dwóch obwodów zamiast jednego, aby obliczyć $\langle O \rangle$. Wraz ze wzrostem liczby składników w operatorze istnieje możliwość, że wymagana liczba wykonań obwodu również wzrośnie.

Rozmiar operatora można ograniczyć, podając argument `operator_budget` funkcji `backpropagate`, który przyjmuje instancję [OperatorBudget](https://docs.quantum.ibm.com/api/qiskit-addon-obp/utils-simplify#operatorbudget).

Aby kontrolować ilość dodatkowych zasobów (liczba wykonań obwodów, a co za tym idzie wymagany czas QPU) przydzielonych do obliczeń, ograniczamy maksymalną liczbę grup Paulich komutujących qubitowo, którą może mieć propagowana wstecznie obserwowalna. Tutaj określamy, że wsteczna propagacja powinna się zatrzymać, gdy liczba qubitowo komutujących grup Paulich w operatorze przekroczy osiem.

In [8]:
# Backpropagate slices onto the observable
bp_obs, remaining_slices, metadata = backpropagate(
    observable, slices, operator_budget=op_budget
)
# Recombine the slices remaining after backpropagation
bp_circuit = combine_slices(remaining_slices)

print(f"Backpropagated {metadata.num_backpropagated_slices} slices.")
print(
    f"New observable has {len(bp_obs.paulis)} terms, which can be combined into {len(bp_obs.group_commuting(qubit_wise=True))} groups."
)
print(
    f"Note that backpropagating one more slice would result in {metadata.backpropagation_history[-1].num_paulis[0]} terms "
    f"across {metadata.backpropagation_history[-1].num_qwc_groups} groups."
)
print("The remaining circuit after backpropagation looks as follows:")
bp_circuit.draw("mpl", fold=-1, scale=0.6)

Backpropagated 6 slices.
New observable has 60 terms, which can be combined into 6 groups.
Note that backpropagating one more slice would result in 114 terms across 12 groups.
The remaining circuit after backpropagation looks as follows:


<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/ee8fd385-1.avif" alt="Output of the previous code cell" />

#### Wsteczne propagowanie przedziałów z obwodu
Najpierw określamy obserwowalną jako $M_Z = \frac{1}{N} \sum_{i=1}^N \langle Z_i \rangle$, gdzie $N$ jest liczbą qubitów. Będziemy propagować wstecz przedziały z obwodu ewolucji czasowej do momentu, gdy składniki obserwowalnej nie będą mogły być już połączone w osiem lub mniej qubitowo komutujących grup Paulich.

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)
print(backend)

<IBMBackend('ibm_kingston')>


In [10]:
pm_basis = generate_preset_pass_manager(
    optimization_level=3, basis_gates=backend.configuration().basis_gates
)
isa_circuit = pm_basis.run(circuit)
isa_bp_circuit = pm_basis.run(bp_circuit)

Poniżej zobaczysz, że propagowaliśmy wstecz sześć przedziałów, a składniki zostały połączone w sześć, a nie osiem grup. Oznacza to, że propagowanie wstecz jeszcze jednego przedziału spowodowałoby przekroczenie ośmiu grup Paulich. Możemy to zweryfikować, analizując zwrócone metadane. Zauważ też, że w tej części transformacja obwodu jest dokładna. To znaczy, żaden ze składników nowej obserwowalnej $O'$ nie został obcięty. Propagowany wstecznie obwód i propagowana wstecznie obserwowalna dają dokładnie taki sam wynik jak pierwotny obwód i obserwowalna.

In [11]:
pubs = [(isa_circuit, observable), (isa_bp_circuit, bp_obs)]

In [12]:
rng = np.random.default_rng()
estimator = StatevectorEstimator(seed=rng)
job = estimator.run(pubs)

![Wyjście poprzedniej komórki kodu](../docs/images/tutorials/operator-back-propagation/extracted-outputs/ee8fd385-1.avif)

W przypadku małoskalowego przykładu na symulatorze nie użyjemy obcinania. Wynika to z faktu, że przy braku szumów obwód z wsteczną propagacją i bez niej prowadzi do tego samego wyniku, a obcinanie pogarsza wynik ze względu na dodaną aproksymację.
#### Transpilacja obwodów do docelowego zestawu bramek
Teraz transpilujemy zarówno pierwotny, jak i propagowany wstecznie obwód do bramek bazowych backendu. Nie musimy transpilować na rzeczywistym backendzie, ponieważ będziemy uruchamiać na symulatorze dla małej instancji.

In [13]:
primitive_result = job.result()
circuit_expval = primitive_result[0].data.evs.item()
bp_circuit_expval = primitive_result[1].data.evs.item()

In [14]:
methods = [
    "No backpropagation",
    "Backpropagation",
]
values = [circuit_expval, bp_circuit_expval]

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
ax.set_ylim([0.6, 0.92])
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/fb5f955a-1.avif" alt="Output of the previous code cell" />

As expected, the two expectation values agree. Because we are running on a noiseless statevector simulator, backpropagation is an exact transformation of the circuit-observable pair, so the original and backpropagated workflows must produce the same value of $M_Z$. The benefit of backpropagation only becomes apparent on noisy hardware, where the shorter backpropagated circuit accumulates less error, as illustrated in the large-scale hardware example below.

## Large-scale hardware example

When developing an experiment, it's useful to start with a small circuit to make visualizations and simulations easier. Now we look into operator backpropagation for a 50-qubit Heisenberg Hamiltonian with the same set of values for the $J$ and $h$ parameters and the same observable $M_Z$, but for four Trotter steps. The ideal expectation value at this scale cannot be calculated in a brute force method, so we use a tensor network and obtain the ideal expectation value to be $\simeq 0.89$.

Along with backpropagation, in this large-scale example, we also introduce backpropagation with truncation. Ideally we want to backpropagate as much as possible to reduce the depth of the effective circuit. However, it often leads to a large number of non-commuting terms in the updated observable, increasing the quantum overhead. Therefore, we can eliminate observable terms with small coefficients using a practice called truncation. While truncation allows more propagation by reducing the number of terms in the updated observable, it also introduces some approximation. Therefore, it is necessary to restrict the truncation within some limits so that the approximation error does not overwhelm the reduction in noise obtained from deeper backpropagation.

To restrict the amount of truncation, we allot an error budget for each slice as well as the total error budget over the entire backpropagated circuit using the [`setup_budget`](/docs/api/qiskit-addon-obp/utils-truncating#setup_budget) function. This ensures that the truncation is controlled for each slice as well as for the entire circuit. See also this [guide](https://qiskit.github.io/qiskit-addon-obp/how_tos/truncate_operator_terms.html) for other ways of allocating the budget.

In [15]:
num_qubits = 50
layout = [(i - 1, i) for i in range(1, num_qubits)]

# Instantiate a CouplingMap object
coupling_map = CouplingMap(layout)

hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(np.pi / 8, np.pi / 4, np.pi / 2),
    ext_magnetic_field=(np.pi / 3, np.pi / 6, np.pi / 9),
)

# Generate a time evolution circuit for the Hamiltonian
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=0.2,
    synthesis=LieTrotter(reps=4),
)

# Define the observable to measure
observable = SparsePauliOp.from_sparse_list(
    [("Z", [i], 1 / num_qubits) for i in range(num_qubits)],
    num_qubits,
)

slices = slice_by_depth(circuit, max_slice_depth=1)

# Define the maximum number of qwc groups allowed in the backpropagated observable, and the truncation error budget
op_budget = OperatorBudget(max_qwc_groups=15)
truncation_error_budget = setup_budget(
    max_error_total=0.03, max_error_per_slice=0.005
)

# First backpropagation without truncation
bp_obs, remaining_slices, metadata = backpropagate(
    observable, slices, operator_budget=op_budget
)
bp_circuit = combine_slices(remaining_slices)

# Now backpropagate with truncation, using the same operator budget and the defined truncation error budget
bp_obs_trunc, remaining_slices_trunc, metadata = backpropagate(
    observable,
    slices,
    operator_budget=op_budget,
    truncation_error_budget=truncation_error_budget,
)
bp_circuit_trunc = combine_slices(
    remaining_slices_trunc, include_barriers=False
)

# Now we transpile the original circuit and the two backpropagated circuits, and apply the layout to the corresponding observables
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

isa_circuit = pm.run(circuit)
isa_bp_circuit = pm.run(bp_circuit)
isa_bp_circuit_trunc = pm.run(bp_circuit_trunc)

isa_observable = observable.apply_layout(isa_circuit.layout)
isa_bp_observable = bp_obs.apply_layout(isa_bp_circuit.layout)
isa_bp_observable_trunc = bp_obs_trunc.apply_layout(
    isa_bp_circuit_trunc.layout
)

# Compare the 2-qubit depth of each transpiled circuit to see how much depth backpropagation saved
print(
    f"2-qubit depth without backpropagation: {isa_circuit.depth(lambda x: x.operation.num_qubits == 2)}"
)
print(
    f"2-qubit depth with backpropagation: {isa_bp_circuit.depth(lambda x: x.operation.num_qubits == 2)}"
)
print(
    f"2-qubit depth with backpropagation and truncation: {isa_bp_circuit_trunc.depth(lambda x: x.operation.num_qubits == 2)}"
)

pubs = [
    (isa_circuit, isa_observable),
    (isa_bp_circuit, isa_bp_observable),
    (isa_bp_circuit_trunc, isa_bp_observable_trunc),
]

# Now we instantiate the Estimator primitive for the hardware with ZNE and measurement error mitigation
# and compute the three circuits and observables
options = EstimatorOptions()
options.default_precision = 0.01
options.resilience_level = 2
options.resilience.zne.noise_factors = [1, 1.2, 1.4]
options.resilience.zne.extrapolator = ["linear"]
estimator = EstimatorV2(mode=backend, options=options)

estimator.options.environment.job_tags = ["TUT_OBP"]
job = estimator.run(pubs)

# Retrieve the results and the standard deviations
result_no_bp = job.result()[0].data.evs.item()
result_bp = job.result()[1].data.evs.item()
result_bp_trunc = job.result()[2].data.evs.item()

std_no_bp = job.result()[0].data.stds.item()
std_bp = job.result()[1].data.stds.item()
std_bp_trunc = job.result()[2].data.stds.item()

2-qubit depth without backpropagation: 24
2-qubit depth with backpropagation: 20
2-qubit depth with backpropagation and truncation: 18


In [16]:
print(f"Expectation value without backpropagation: {result_no_bp}")
print(f"Backpropagated expectation value: {result_bp}")
print(f"Backpropagated expectation value with truncation: {result_bp_trunc}")

Expectation value without backpropagation: 0.9543907942381811
Backpropagated expectation value: 0.9445337385406468
Backpropagated expectation value with truncation: 0.934050286970965


In [17]:
# Plot the results
methods = [
    "No backpropagation",
    "Backpropagation",
    "Backpropagation w/ truncation",
]
values = [result_no_bp, result_bp, result_bp_trunc]
error_bars = [std_no_bp, std_bp, std_bp_trunc]

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.errorbar(methods, values, yerr=error_bars, fmt="o", color="r", capsize=5)
plt.axhline(0.89)
ax.set_ylim([0.8, 0.98])
plt.text(0.25, 0.895, "Exact result")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/37834c72-1.avif" alt="Output of the previous code cell" />

### Krok 4: Post-processing i zwrócenie wyników w żądanym formacie klasycznym
Teraz uzyskujemy wartości oczekiwane pierwotnego i propagowanego wstecznie obwodu.